In [1]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では安全に無視され、Colab環境でのみ自動実行されます
import os, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_name = "study-mikanos"
    repo_url = "https://github.com/miyayudai/study-mikanos.git"
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"{repo_name} リポジトリをダウンロード中...")
        !git clone {repo_url} /content/{repo_name}
    os.chdir(f"/content/{repo_name}/2")
    sys.path.insert(0, f"/content/{repo_name}")
    !pip install -q -r /content/{repo_name}/requirements.txt
    !pip install -q -e /content/{repo_name}
    print("環境セットアップが完了しました。作業ディレクトリ:", os.getcwd())
else:
    # ローカル環境の場合、プロジェクトルートをパスに追加
    proj_root = Path.cwd().resolve()
    if proj_root.name == "2":
        proj_root = proj_root.parent
    if str(proj_root) not in sys.path:
        sys.path.insert(0, str(proj_root))
    print("ローカル実行環境を検出しました。プロジェクトルート:", proj_root)


ローカル実行環境を検出しました。プロジェクトルート: /home/student/Documents/GitHub/study-mikanos


# 第2章 EDK II入門とメモリマップ — 2.1 EDK II入門

本節では、名著『**ゼロからのOS自作入門**』（内田公太 著、マイナビ出版）の **第2章 2.1「EDK II入門」** に基づき、UEFI アプリケーションおよび OS ブートローダを開発するための公式標準開発フレームワークである **EDK II (EFI Development Kit II)** のアーキテクチャ、パッケージシステム、およびビルドメカニズムを徹底解説します。

---

## 🎯 本節の学習目標と技術的動機

1. **第1章の限界と EDK II 導入の必然性を理解する**:
   - 第1章（1.9節）では、手書きの最小構造体（`char dummy[52];`）と Clang/LLD の直接呼び出しで `hello.efi` を作成しました。
   - しかし、OS ブートローダとして必要な **メモリマップ取得 (`GetMemoryMap`)**、**画面描画 (`GraphicsOutputProtocol`)**、**ファイル読み書き (`SimpleFileSystemProtocol`)**、**ACPI テーブル取得** などの膨大な UEFI 仕様 API をすべて手書きで保守することは不可能です。
   - UEFI 仕様を策定する UEFI Forum および Intel のオープンソースリファレンス実装である **EDK II** を導入する技術的必然性を掴みます。
2. **EDK II のパッケージ階層構造を把握する**:
   - UEFI 標準仕様のヘッダ・ライブラリを提供する **`MdePkg` (Module Development Environment Package)** と、自作ブートローダを定義する **`MikanLoaderPkg`** の関係を理解します。
3. **パッケージシステム3大メタ定義ファイル形式を完全解剖する**:
   - **DEC (`.dec`, Package Declaration File)**: パッケージが外部に公開するインターフェース宣言（公開ヘッダ、GUID、プロトコル）。
   - **INF (`.inf`, Module Information File)**: 単一モジュール（実行ファイルやライブラリ）の構成定義（ソース、エントリポイント、抽象ライブラリ要求）。
   - **DSC (`.dsc`, Platform Description File)**: プラットフォーム全体のビルド構成と、**依存性注入 (Dependency Injection)** による抽象ライブラリと具象実装の結合。
4. **EDK II BaseTools による 6段階ビルドパイプラインを習得する**:
   - `build` コマンドがどのように DSC を解析し、AutoGen フェーズでエントリポイントラッパーや Makefile を自動合成し、最終的な PE32+ バイナリ（`Loader.efi`）を生成するのかをシミュレーションを通じて実証します。


## 1. なぜ自作OS開発に EDK II が必要なのか？

### 1.1 第1章の手書き Freestanding C方式が抱えていた限界
第1章 1.9節では、外部ヘッダに一切依存しない Freestanding C言語により、以下のように自前で構造体を定義して文字を出力しました：

```c
// 第1章 1.9節 hello.c の手書き構造体
struct EFI_SYSTEM_TABLE {
    char dummy[52];
    void* ConsoleOutHandle;
    struct EFI_SIMPLE_TEXT_OUTPUT_PROTOCOL* ConOut;
};
```

このアプローチは「最小限のバイナリが動く仕組み」を学ぶ上では極めて強力でしたが、実用的な OS ブートローダを開発する上では直ちに以下の致命的な壁に衝突します：

1. **仕様定義の爆発的増大**:
   - UEFI 2.8 仕様書は **2,500ページ以上** に及びます。メモリ管理関数、ファイル入出力プロトコル、画面描画プロトコル（GOP）、デバイスパス、PCI バス操作などの構造体・GUID・定数をすべて手動で定義するのは非現実的です。
2. **パディング・アライメントと ABI 不整合のリスク**:
   - 構造体のパディングやアライメント（64ビット環境での自然境界整列）を 1 バイトでも間違えると、ファームウェアがクラッシュ（Triple Fault / ハングアップ）します。
3. **エントリポイント初期化の隠蔽**:
   - UEFI アプリケーションは、単に `main` 関数を実行するだけでなく、標準グローバルポインタ（`gST` = System Table, `gBS` = Boot Services, `gImageHandle` = 自モジュールハンドル）の初期化やスタック保護のセットアップが必要です。

---

### 1.2 EDK II (EFI Development Kit II) とは？
**EDK II** は、TianoCore プロジェクト（Intel が主導する UEFI オープンソースコミュニティ）によって保守されている、UEFI 仕様のクロスプラットフォーム・リファレンス実装および開発環境です。

- **仕様準拠性**: 最新の UEFI 仕様および PI (Platform Initialization) 仕様に 100% 準拠した標準ヘッダ群（`Uefi.h`, `PiPei.h`, `PiDxe.h` 等）を提供。
- **豊富な標準ライブラリ**: `Print()`, `CopyMem()`, `SetMem()` などの安全なユーティリティ関数（`UefiLib`, `BaseMemoryLib`）を完備。
- **強固なモジュール化アーキテクチャ**: 各種ファームウェアコンポーネント（ドライバ、ローダ、ライブラリ）を独立したモジュール（INF）として再利用可能。


## 2. EDK II のパッケージ階層構造とモジュール設計

EDK II の世界では、すべてのコードは **Package（パッケージ）** と **Module（モジュール）** という階層的な単位で管理されます。

```
edk2/ (WORKSPACE)
 ├── MdePkg/                     # 標準パッケージ (Module Development Environment)
 │    ├── MdePkg.dec              # 標準仕様ヘッダ・GUID の公開宣言
 │    ├── Include/                # Uefi.h, Library/UefiLib.h 等
 │    └── Library/                # 標準ライブラリの具象実装 (INF)
 └── MikanLoaderPkg/             # 本書で作成するオリジナル OS ブートローダ
      ├── MikanLoaderPkg.dec      # パッケージ宣言
      ├── MikanLoaderPkg.dsc      # プラットフォーム記述・依存性注入
      ├── Loader.inf              # ブートローダモジュール定義
      └── Main.c                  # UefiMain 実装
```

以下の図は、EDK II におけるパッケージ階層と、後述する 3大メタ定義ファイル（DEC, INF, DSC）がどのように協調して依存関係を解決（Dependency Injection）するかを示したアーキテクチャ図です。

<div align="center">
  <img src="result/fig2_01_edk2_architecture_and_packages.png" width="90%" alt="EDK II Package Architecture">
  <p><b>Figure 2.1: EDK II パッケージ階層と依存性逆転モデル (Dependency Inversion Model)</b></p>
</div>


## 3. パッケージシステム3大メタ定義ファイルの完全解剖

EDK II のビルドシステムは、Make や CMake のような一般的なビルドツールとは異なり、**「宣言（DEC）」「要求（INF）」「結合（DSC）」** という 3 つの関心事に厳密に分離されています。

---

### 3.1 DEC (`.dec`, Package Declaration File) — パッケージの公開インターフェース宣言
DEC ファイルは、**「このパッケージは世界（他のパッケージ）に対して何を公開・提供できるか」** を宣言する仕様書です。

```ini
# MikanLoaderPkg/MikanLoaderPkg.dec
[Defines]
  DEC_SPECIFICATION              = 0x00010005
  PACKAGE_NAME                   = MikanLoaderPkg
  PACKAGE_GUID                   = 452eae8e-71e9-11e8-a243-df3f1ffdebe1
  PACKAGE_VERSION                = 0.1

[Includes]
  Include                        # 公開インクルードディレクトリ
```

- `[Defines]`: パッケージ名、GUID（全世界で一意な128ビット識別子）、バージョン。
- `[Includes]`: 他のモジュールから `#include <...>` できる公開インクルードパス。
- `[Guids]`, `[Protocols]`: このパッケージが定義する UEFI プロトコルの GUID。

---

### 3.2 INF (`.inf`, Module Information File) — モジュールのビルド仕様と要求
INF ファイルは、単一の実行可能ファイル（`.efi`）やライブラリ（`.lib`）のビルド仕様書です。
最大の特徴は、**「具体的なライブラリ実装のファイルパスではなく、抽象ライブラリクラス名（Library Class）のみを要求する」** という点です。

```ini
# MikanLoaderPkg/Loader.inf
[Defines]
  INF_VERSION                    = 0x00010006
  BASE_NAME                      = Loader
  FILE_GUID                      = c9d0d202-71e9-11e8-9e52-cfbfd0063fbf
  MODULE_TYPE                    = UEFI_APPLICATION
  VERSION_STRING                 = 0.1
  ENTRY_POINT                    = UefiMain

[Sources]
  Main.c                         # コンパイル対象ソースコード

[Packages]
  MdePkg/MdePkg.dec              # 参照するパッケージ宣言

[LibraryClasses]
  UefiLib                        # 抽象要求: Print() 等の UEFI ユーティリティ
  UefiApplicationEntryPoint      # 抽象要求: UefiMain を呼び出すエントリポイント
```

- `MODULE_TYPE = UEFI_APPLICATION`: このモジュールが UEFI アプリケーション（Ring 0 で動作する一時的プログラム）であることを指定。ドライバの場合は `DXE_DRIVER` などになります。
- `ENTRY_POINT = UefiMain`: ファームウェアから最初に制御が渡される関数のシンボル名。
- `[LibraryClasses]`: このモジュールが必要とする **抽象インターフェース名**。どの具象ファイルを使うかはここでは一切決めません！

---

### 3.3 DSC (`.dsc`, Platform Description File) — 依存性注入とプラットフォーム統合
DSC ファイルは、**「どのモジュールをビルドし、抽象ライブラリクラスをどの具象実装に結びつけるか」** を決定する最高意思決定ファイルです。ソフトウェア工学における **依存性注入 (Dependency Injection / DI)** をビルドシステムレベルで実現しています。

```ini
# MikanLoaderPkg/MikanLoaderPkg.dsc
[Defines]
  PLATFORM_NAME                  = MikanLoaderPkg
  PLATFORM_GUID                  = d3f11f4e-71e9-11e8-a7e1-33fd4f7d5a3e
  PLATFORM_VERSION               = 0.1
  DSC_SPECIFICATION              = 0x00010005
  OUTPUT_DIRECTORY               = Build/MikanLoader$(ARCH)
  SUPPORTED_ARCHITECTURES        = X64
  BUILD_TARGETS                  = DEBUG|RELEASE|NOOPT

[LibraryClasses]
  # 抽象インターフェース名 | 具象実装ライブラリの INF パス
  UefiApplicationEntryPoint|MdePkg/Library/UefiApplicationEntryPoint/UefiApplicationEntryPoint.inf
  UefiLib|MdePkg/Library/UefiLib/UefiLib.inf
  BaseLib|MdePkg/Library/BaseLib/BaseLib.inf
  BaseMemoryLib|MdePkg/Library/BaseMemoryLib/BaseMemoryLib.inf
  DebugLib|MdePkg/Library/BaseDebugLibNull/BaseDebugLibNull.inf
  DevicePathLib|MdePkg/Library/UefiDevicePathLib/UefiDevicePathLib.inf
  MemoryAllocationLib|MdePkg/Library/UefiMemoryAllocationLib/UefiMemoryAllocationLib.inf
  PcdLib|MdePkg/Library/BasePcdLibNull/BasePcdLibNull.inf
  PrintLib|MdePkg/Library/BasePrintLib/BasePrintLib.inf
  UefiBootServicesTableLib|MdePkg/Library/UefiBootServicesTableLib/UefiBootServicesTableLib.inf
  UefiRuntimeServicesTableLib|MdePkg/Library/UefiRuntimeServicesTableLib/UefiRuntimeServicesTableLib.inf

[Components]
  MikanLoaderPkg/Loader.inf      # ビルド対象コンポーネント
```

> [!TIP]
> **なぜ抽象化するのか？（具象ライブラリ差し替えの威力）**:
> 例えばデバッグ出力用の `DebugLib` を、実機開発時にはシリアルポート出力版（`BaseDebugLibSerialPort`）に、エミュレータ時には画面出力版に、テスト時には何もしない Null スタブ（`BaseDebugLibNull`）に、**C言語のソースコード（`Main.c`）を 1行も書き換えることなく、DSC の 1行を変更するだけで一括切替** できます。


In [2]:
# === Python シミュレータによる DEC / INF / DSC のメタデータ解析 ===
from common.os_sim import (
    Edk2DecModel,
    Edk2InfModel,
    Edk2DscModel,
    DscDependencyResolver,
    Edk2BuildPipelineSimulator,
    BuildApproachComparator,
)

# 1. MikanLoaderPkg.dec のパース
dec_text = """
[Defines]
  DEC_SPECIFICATION              = 0x00010005
  PACKAGE_NAME                   = MikanLoaderPkg
  PACKAGE_GUID                   = 452eae8e-71e9-11e8-a243-df3f1ffdebe1
  PACKAGE_VERSION                = 0.1
"""
dec = Edk2DecModel.parse_dec_text(dec_text)
print("【DEC パース結果】")
print(f"  Package Name: {dec.package_name}")
print(f"  Package GUID: {dec.package_guid}")
print(f"  Version:      {dec.package_version}")

# 2. Loader.inf のパース
inf_text = """
[Defines]
  INF_VERSION                    = 0x00010006
  BASE_NAME                      = Loader
  FILE_GUID                      = c9d0d202-71e9-11e8-9e52-cfbfd0063fbf
  MODULE_TYPE                    = UEFI_APPLICATION
  VERSION_STRING                 = 0.1
  ENTRY_POINT                    = UefiMain

[Sources]
  Main.c

[Packages]
  MdePkg/MdePkg.dec

[LibraryClasses]
  UefiLib
  UefiApplicationEntryPoint
"""
inf = Edk2InfModel.parse_inf_text(inf_text)
print("\n【INF パース結果】")
print(f"  Module BaseName: {inf.base_name}")
print(f"  Module Type:     {inf.module_type}")
print(f"  Entry Point:     {inf.entry_point}")
print(f"  Required Libs:   {inf.library_classes}")

# 3. MikanLoaderPkg.dsc のパース
dsc_text = """
[Defines]
  PLATFORM_NAME                  = MikanLoaderPkg
  PLATFORM_GUID                  = d3f11f4e-71e9-11e8-a7e1-33fd4f7d5a3e
  PLATFORM_VERSION               = 0.1
  DSC_SPECIFICATION              = 0x00010005
  OUTPUT_DIRECTORY               = Build/MikanLoader$(ARCH)
  SUPPORTED_ARCHITECTURES        = X64
  BUILD_TARGETS                  = DEBUG|RELEASE|NOOPT

[LibraryClasses]
  UefiApplicationEntryPoint|MdePkg/Library/UefiApplicationEntryPoint/UefiApplicationEntryPoint.inf
  UefiLib|MdePkg/Library/UefiLib/UefiLib.inf
  BaseLib|MdePkg/Library/BaseLib/BaseLib.inf
  BaseMemoryLib|MdePkg/Library/BaseMemoryLib/BaseMemoryLib.inf
  DebugLib|MdePkg/Library/BaseDebugLibNull/BaseDebugLibNull.inf
  DevicePathLib|MdePkg/Library/UefiDevicePathLib/UefiDevicePathLib.inf
  MemoryAllocationLib|MdePkg/Library/UefiMemoryAllocationLib/UefiMemoryAllocationLib.inf
  PcdLib|MdePkg/Library/BasePcdLibNull/BasePcdLibNull.inf
  PrintLib|MdePkg/Library/BasePrintLib/BasePrintLib.inf
  UefiBootServicesTableLib|MdePkg/Library/UefiBootServicesTableLib/UefiBootServicesTableLib.inf
  UefiRuntimeServicesTableLib|MdePkg/Library/UefiRuntimeServicesTableLib/UefiRuntimeServicesTableLib.inf

[Components]
  MikanLoaderPkg/Loader.inf
"""
dsc = Edk2DscModel.parse_dsc_text(dsc_text)
print("\n【DSC パース結果】")
print(f"  Platform Name:   {dsc.platform_name}")
print(f"  Architectures:   {dsc.supported_architectures}")
print(f"  Target Modes:    {dsc.build_targets}")
print(f"  Bound Libraries: {len(dsc.library_classes)} classes configured")
print(f"  Components:      {dsc.components}")


【DEC パース結果】
  Package Name: MikanLoaderPkg
  Package GUID: 452eae8e-71e9-11e8-a243-df3f1ffdebe1
  Version:      0.1

【INF パース結果】
  Module BaseName: Loader
  Module Type:     UEFI_APPLICATION
  Entry Point:     UefiMain
  Required Libs:   ['UefiLib', 'UefiApplicationEntryPoint']

【DSC パース結果】
  Platform Name:   MikanLoaderPkg
  Architectures:   ['X64']
  Target Modes:    ['DEBUG', 'RELEASE', 'NOOPT']
  Bound Libraries: 11 classes configured
  Components:      ['MikanLoaderPkg/Loader.inf']


In [3]:
# === 依存性注入（Dependency Inversion / Injection）解決シミュレーション ===
resolver = DscDependencyResolver(dsc=dsc)
resolver.register_module("MikanLoaderPkg/Loader.inf", inf)

# 正常系の解決テスト
res = resolver.resolve_dependencies("MikanLoaderPkg/Loader.inf")
print("【依存関係解決レポート (正常系)】")
print(f"  ターゲットモジュール: {res['component']}")
print(f"  解決状態 (is_valid):  {res['is_valid']}")
print(f"  要求ライブラリ数:     {res['total_required']}")
print(f"  解決済みインスタンス: {res['total_resolved']}")
for req, bound in res['resolved'].items():
    print(f"    * 抽象 [{req}] -> 具象 [{bound}]")

# 異常系の解決テスト（未知のライブラリが要求された場合）
print("\n【異常系テスト: 未知のライブラリクラス要求の検出】")
bad_inf = Edk2InfModel.parse_inf_text(inf_text)
bad_inf.library_classes.append("CustomGraphicsAccelerationLib")
resolver.register_module("MikanLoaderPkg/Loader_Bad.inf", bad_inf)

bad_res = resolver.resolve_dependencies("MikanLoaderPkg/Loader_Bad.inf")
print(f"  解決状態 (is_valid):  {bad_res['is_valid']}")
print(f"  未解決ライブラリ:     {bad_res['missing']}")
assert not bad_res['is_valid'], "未解決依存が正しく検出される必要があります"
print("  ✓ 期待通り、未解決エラーが検知されました！")


【依存関係解決レポート (正常系)】
  ターゲットモジュール: MikanLoaderPkg/Loader.inf
  解決状態 (is_valid):  True
  要求ライブラリ数:     2
  解決済みインスタンス: 2
    * 抽象 [UefiLib] -> 具象 [MdePkg/Library/UefiLib/UefiLib.inf]
    * 抽象 [UefiApplicationEntryPoint] -> 具象 [MdePkg/Library/UefiApplicationEntryPoint/UefiApplicationEntryPoint.inf]

【異常系テスト: 未知のライブラリクラス要求の検出】
  解決状態 (is_valid):  False
  未解決ライブラリ:     ['CustomGraphicsAccelerationLib']
  ✓ 期待通り、未解決エラーが検知されました！


## 4. EDK II BaseTools による 6段階ビルドパイプライン

第1章のビルドはシェルから `clang` と `lld-link` を直接叩く 1ステップ（あるいはオブジェクト生成を挟む 2ステップ）のシンプルなものでした。
一方、EDK II のビルドシステムは **BaseTools** と呼ばれる Python / C言語ツール群によって駆動され、以下の 6段階の厳密なパイプラインを辿ります。

<div align="center">
  <img src="result/fig2_02_edk2_build_pipeline_comparison.png" width="95%" alt="Build Pipeline Comparison">
  <p><b>Figure 2.2: 第1章 (手動Clang/LLD) vs 第2章 (EDK II BaseTools) ビルドパイプライン比較</b></p>
</div>

### 6段階パイプラインの詳細：
1. **Environment Validation (`edksetup.sh`)**:
   - `WORKSPACE`, `EDK_TOOLS_PATH`, `CONF_PATH` などの環境変数を検証し、BaseTools の実行パスをセットアップ。
2. **Platform & Target Resolution (`build` コマンド)**:
   - `-p MikanLoaderPkg/MikanLoaderPkg.dsc -a X64 -b DEBUG -t CLANG38` などの引数から、対象プラットフォーム・CPUアーキテクチャ・ツールチェーンを確定。
3. **Library Dependency Injection**:
   - 各 INF が要求する LibraryClasses を DSC のマッピングテーブルと照合し、具象ライブラリインスタンスを再帰的に解決。
4. **AutoGen Synthesizer (`build.py`)**:
   - C言語のソースコード（`AutoGen.c`, `AutoGen.h`）および各モジュール用の `Makefile` を自動生成。
   - `AutoGen.c` 内に `ProcessLibraryConstructorList()` やファームウェア初期化用のエントリポイントラッパー（`_ModuleEntryPoint`）が合成されます。
5. **Compile & Link**:
   - 生成された Makefile に基づき、Clang で C コードを COFF オブジェクト（`.obj`）へコンパイルし、LLD-Link で DLL / PE 形式へリンク。
6. **GenFw Image Packaging**:
   - EDK II 独自の `GenFw` ツールにより、PE32+ ヘッダの規格厳格チェック、再配置テーブル（`.reloc`）の調整を行い、最終的な `Loader.efi` を出力。


In [4]:
# === EDK II 6段階ビルドパイプラインのシミュレーション実行 ===
sim = Edk2BuildPipelineSimulator(
    workspace="/home/student/edk2",
    edk_tools_path="/home/student/edk2/BaseTools",
    target_arch="X64",
    target_toolchain="CLANG38",
    build_target="DEBUG"
)

build_result = sim.simulate_build(dsc, inf)

print("【EDK II ビルドパイプライン実行ログ】")
print(f"ビルド成否: {'SUCCESS' if build_result['success'] else 'FAILED'}")
print(f"最終生成バイナリ: {build_result['final_efi']}\n")

for step in build_result["steps"]:
    status_icon = "✓" if step["status"] == "PASS" else "✗"
    print(f"[{status_icon}] Step {step['step']}: {step['name']}")
    print(f"    コマンド/処理: {step['command']}")
    print(f"    詳細情報:     {step['detail']}")
    if "generated_files" in step:
        for f in step["generated_files"]:
            print(f"      - {f}")
    print()


【EDK II ビルドパイプライン実行ログ】
ビルド成否: SUCCESS
最終生成バイナリ: Build/MikanLoaderX64/DEBUG_CLANG38/X64/MikanLoaderPkg/Loader/Loader.efi

[✓] Step 1: Environment Validation
    コマンド/処理: edksetup.sh
    詳細情報:     WORKSPACE=/home/student/edk2, EDK_TOOLS_PATH=/home/student/edk2/BaseTools

[✓] Step 2: Platform & Target Resolution
    コマンド/処理: build -p MikanLoaderPkg.dsc -a X64 -b DEBUG -t CLANG38
    詳細情報:     Target Arch: X64 (Supported: True), Mode: DEBUG (Supported: True)

[✓] Step 3: Library Dependency Resolution
    コマンド/処理: Dependency Injection (LibraryClasses -> Instances)
    詳細情報:     Required 2 classes, Resolved 2 instances (Missing: [])

[✓] Step 4: AutoGen Synthesizer
    コマンド/処理: BaseTools/Bin/build.py (AutoGen phase)
    詳細情報:     Generated 3 AutoGen artifacts (Entry vector, GUID descriptors, dependency Makefile)
      - Build/MikanLoaderX64/DEBUG_CLANG38/X64/MikanLoaderPkg/Loader/AutoGen.c
      - Build/MikanLoaderX64/DEBUG_CLANG38/X64/MikanLoaderPkg/Loader/AutoGen.h
      - Build/MikanLoade

In [5]:
# === 第1章 (手動Clang) vs 第2章 (EDK II) の多角性技術比較 ===
import pandas as pd

comparator = BuildApproachComparator()
df_compare = pd.DataFrame(comparator.criteria)

# 表形式で見やすく表示
pd.set_option('display.max_colwidth', None)
print("【第1章 vs 第2章 ビルド方式の徹底比較マトリクス】")
display(df_compare.rename(columns={
    "category": "比較項目",
    "ch1_manual": "第1章: 手動Clang直接ビルド",
    "ch2_edk2": "第2章: EDK II BaseTools",
    "advantage": "EDK II の技術的利点"
}))


【第1章 vs 第2章 ビルド方式の徹底比較マトリクス】


,比較項目,第1章: 手動Clang直接ビルド,第2章: EDK II BaseTools,EDK II の技術的利点
0,ツールチェーン & 駆動方式,Clang + LLD をシェルから直接ワンライナー実行,Python/C による BaseTools (`build`) 駆動のメタビルド,EDK II は数千ファイルの巨大ファームウェアでも破綻しない
1,UEFI 仕様準拠と標準ヘッダ,必要な構造体（`EFI_SYSTEM_TABLE` 等）と型を手書き定義,`MdePkg` (Module Dev Env) による全仕様ヘッダの完全提供,"EDK II は仕様変更や拡張プロトコル（GOP, FS, USB）に即応可能"
2,ライブラリ抽象化 (DI),なし（関数ポインタ直接呼び出し）,LibraryClass 抽象インターフェースと DSC での実装注入,実機デバッグ用とNULLスタブ用ライブラリをDSC一行で切替可能
3,マルチアーキテクチャ移植性,x86-64 専用フラグをハードコード,"DSC/INF で X64, IA32, ARM, AARCH64, RISCV64 を透過サポート",CPUアーキテクチャ非依存の共通ローダ設計が可能
4,学習コストと初期設定,極めて低い（LLVMさえあれば1秒で実行可能）,高い（BaseToolsビルド、環境変数、DSC/INF構文の習得が必要）,第1章は入門に最適、第2章以降の実践OS開発にはEDK IIが必須


In [6]:
# === C++20 スタンドアロン検証ツール (ch02_edk2_package_analyzer) の実行 ===
import subprocess
from pathlib import Path

# プロジェクトルートからのバイナリパス解決
bin_path = Path("../build/bin/ch02_edk2_package_analyzer")
if not bin_path.exists():
    bin_path = Path("build/bin/ch02_edk2_package_analyzer")
if not bin_path.exists():
    bin_path = Path("../bin/ch02_edk2_package_analyzer")

if bin_path.exists():
    print(f"C++20 バイナリを実行中: {bin_path}\n")
    proc = subprocess.run([str(bin_path)], capture_output=True, text=True, check=True)
    print(proc.stdout)
else:
    print("⚠️ C++20 バイナリが見つかりません。cmake --build build を実行してください。")


C++20 バイナリを実行中: ../build/bin/ch02_edk2_package_analyzer



 MikanOS EDK II Architecture & Package Analyzer (C++20 Standalone)

[Test 1] Parsing MikanLoaderPkg.dec ...
  ✓ Package Name: MikanLoaderPkg
  ✓ Package GUID: 452eae8e-71e9-11e8-a243-df3f1ffdebe1
  ✓ DEC Spec:     0x00010005

[Test 2] Parsing Loader.inf ...
  ✓ Base Name:    Loader
  ✓ Module Type:  UEFI_APPLICATION
  ✓ Entry Point:  UefiMain
  ✓ Sources:      Main.c
  ✓ Required Libs:2 classes

[Test 3] Parsing MikanLoaderPkg.dsc ...
  ✓ Platform:     MikanLoaderPkg
  ✓ Bound Libs:   11 instances
  ✓ Components:   1 module(s)

[Test 4] Resolving Loader.inf dependencies via MikanLoaderPkg.dsc ...
  ✓ Dependency Resolution: PASS (All 2 classes resolved)
      * Class [UefiApplicationEntryPoint] -> Instance [MdePkg/Library/UefiApplicationEntryPoint/UefiApplicationEntryPoint.inf]
      * Class [UefiLib] -> Instance [MdePkg/Library/UefiLib/UefiLib.inf]

[Test 5] Negative Test: Unresolved Library Class Detection ...
  ✓ Unresolved Dependency correctly caught: NonExistentSpecialDriverLib

 🎉

In [7]:
# === 第2章 2.1節 理解度確認クイズ ===
# 各質問の回答を選択肢の番号（1〜4）で入力してください。

answers = {}

print("Q1: EDK II において、パッケージが外部に公開するインターフェース（ヘッダやGUID）を宣言するファイル形式は？")
print("  [1] .inf")
print("  [2] .dec")
print("  [3] .dsc")
print("  [4] .fdf")
answers["Q1"] = 2  # 正解: .dec

print("\nQ2: モジュール定義ファイル (.inf) の [LibraryClasses] セクションに記述する内容として正しいものは？")
print("  [1] 具象ライブラリの .inf ファイルへの絶対パス")
print("  [2] 抽象ライブラリクラス名（例: UefiLib）")
print("  [3] コンパイル対象の C 言語ソースコード")
print("  [4] ライブラリのバイナリコード (.lib)")
answers["Q2"] = 2  # 正解: 抽象ライブラリクラス名

print("\nQ3: プラットフォーム定義ファイル (.dsc) が果たしている設計パターンの役割は何か？")
print("  [1] シングルトンパターン (Singleton)")
print("  [2] オブザーバーパターン (Observer)")
print("  [3] 依存性注入 (Dependency Injection / DI)")
print("  [4] ファクトリメソッド (Factory Method)")
answers["Q3"] = 3  # 正解: 依存性注入

print("\nQ4: EDK II のビルドパイプラインにおいて、AutoGen フェーズが自動合成するものはどれか？")
print("  [1] AutoGen.c, AutoGen.h, およびモジュール用 Makefile")
print("  [2] 最終的な Loader.efi 実行バイナリ")
print("  [3] GCC や Clang などのコンパイラ本体")
print("  [4] UEFI ファームウェア ROM イメージ")
answers["Q4"] = 1  # 正解: AutoGen.c, AutoGen.h, Makefile

print("\nQ5: MikanLoaderPkg において、Loader.inf の MODULE_TYPE に指定すべき種別は？")
print("  [1] SEC")
print("  [2] DXE_DRIVER")
print("  [3] UEFI_APPLICATION")
print("  [4] PEIM")
answers["Q5"] = 3  # 正解: UEFI_APPLICATION

# 採点
score = 0
correct_keys = {"Q1": 2, "Q2": 2, "Q3": 3, "Q4": 1, "Q5": 3}
for q, user_ans in answers.items():
    if user_ans == correct_keys[q]:
        score += 1

print(f"\n採点結果: {score} / {len(correct_keys)} (正解率: {score/len(correct_keys)*100:.1f}%)")
if score == len(correct_keys):
    print("🎉 全問正解です！EDK II のパッケージ構造とビルドアーキテクチャを完璧に理解しました！")


Q1: EDK II において、パッケージが外部に公開するインターフェース（ヘッダやGUID）を宣言するファイル形式は？
  [1] .inf
  [2] .dec
  [3] .dsc
  [4] .fdf

Q2: モジュール定義ファイル (.inf) の [LibraryClasses] セクションに記述する内容として正しいものは？
  [1] 具象ライブラリの .inf ファイルへの絶対パス
  [2] 抽象ライブラリクラス名（例: UefiLib）
  [3] コンパイル対象の C 言語ソースコード
  [4] ライブラリのバイナリコード (.lib)

Q3: プラットフォーム定義ファイル (.dsc) が果たしている設計パターンの役割は何か？
  [1] シングルトンパターン (Singleton)
  [2] オブザーバーパターン (Observer)
  [3] 依存性注入 (Dependency Injection / DI)
  [4] ファクトリメソッド (Factory Method)

Q4: EDK II のビルドパイプラインにおいて、AutoGen フェーズが自動合成するものはどれか？
  [1] AutoGen.c, AutoGen.h, およびモジュール用 Makefile
  [2] 最終的な Loader.efi 実行バイナリ
  [3] GCC や Clang などのコンパイラ本体
  [4] UEFI ファームウェア ROM イメージ

Q5: MikanLoaderPkg において、Loader.inf の MODULE_TYPE に指定すべき種別は？
  [1] SEC
  [2] DXE_DRIVER
  [3] UEFI_APPLICATION
  [4] PEIM

採点結果: 5 / 5 (正解率: 100.0%)
🎉 全問正解です！EDK II のパッケージ構造とビルドアーキテクチャを完璧に理解しました！


## 5. 本節のまとめと次節 2.2 への展望

### 5.1 本節の重要ポイント総括
1. **EDK II 導入の必然性**:
   - 膨大な UEFI 仕様 API（メモリマップ、ファイルシステム、画面描画 GOP）を手書き構造体で保守することは不可能なため、公式フレームワーク EDK II を導入しました。
2. **三位一体のパッケージメタ定義**:
   - **DEC (`.dec`)**: パッケージが何を提供できるか（インターフェース宣言）。
   - **INF (`.inf`)**: 個別モジュールが何を必要とするか（抽象ライブラリクラス要求）。
   - **DSC (`.dsc`)**: プラットフォームが要求と具象実装をどう結びつけるか（依存性注入）。
3. **6段階ビルドパイプライン**:
   - BaseTools の `build` コマンドにより、DSC 解析 → 依存解決 → AutoGen 生成 → Clang コンパイル → LLD リンク → GenFw パッケージングを経て `Loader.efi` が合成されます。

---

### 5.2 次節 2.2「EDK IIでハローワールド」への展望
本節で EDK II のアーキテクチャとメタ定義ファイルの役割を完全に理解しました。
次節 **2.2「EDK IIでハローワールド (`osbook_day02a`)」** では、実際に `Main.c` を記述し、EDK II 標準の `UefiLib` が提供する `Print(L"Hello, Mikan World!\n");` を使って、EDK II 環境下での最初のハローワールドアプリケーションをビルド・実行します！
